# 🌍 Notebook 4: Typeahead in the real world


## 🛠️ Setup

```bash
cd 06-system-designs/typeahead-autocomplete
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## How production systems actually do this

Our lab trie is the core idea behind almost every real autocomplete.
But big systems add layers on top. Quick tour:

### Google Search
- Trie-like in-memory index, sharded across many machines.
- Heavy **personalization**: your past searches, location, language.
- **Trending** (e.g. breaking-news terms surge within minutes).
- Spell correction is first-class: `"gogole"` still suggests `"google"`.

### YouTube Search
- Same shape, but the corpus is video titles + query history.
- Suggestions are also **diversified** (avoid 10 near-duplicate titles).

### Amazon Product Search
- Suggestions are scoped: global + **per-department** (Books vs Electronics).
- Ranking signals include click-through rate and conversion, not just frequency.

### Elasticsearch "completion suggester"
- Backed by an **FST** (Finite State Transducer) — a compressed trie that
  also stores weights on edges. Same O(L) lookup idea.
- Tuning knobs: `size`, `fuzzy.fuzziness`, `contexts` (category filters).

### Redis sorted sets trick (what small teams use)
- Redis has no trie, but `ZADD / ZRANGEBYLEX` lets you do prefix queries on a
  sorted set in O(log N + R) — that's our sorted+bisect version, just in Redis.
- Good enough for tens of millions of terms and a few thousand QPS. No rebuild
  pipeline needed — `ZINCRBY` increments counts live.

We'll reproduce the Redis approach in pure Python below so you can feel it.


## Mini-reproduction: Redis-style prefix range with `bisect`

This is the same algorithm Redis uses for `ZRANGEBYLEX "[prefix" "[prefix\xff"`
on a sorted set, minus the network and persistence. It's the easy production
choice when you don't want to run a trie service.


In [ ]:
# 📦 "Poor-man's Redis": a sorted list supporting prefix range + live ZINCRBY.
import bisect, unicodedata
from heapq import nlargest

def normalize(s: str) -> str:
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    return " ".join(s.lower().strip().split())

class ZSet:
    """A miniature of a Redis sorted set with a prefix-range operation."""
    def __init__(self):
        self._terms: list[str] = []         # sorted
        self._score: dict[str, int] = {}

    def zincrby(self, term: str, by: int = 1):
        term = normalize(term)
        if term not in self._score:
            bisect.insort(self._terms, term)
            self._score[term] = 0
        self._score[term] += by

    def zrangebylex_prefix(self, prefix: str, k: int = 5):
        prefix = normalize(prefix)
        if not prefix:
            return []
        # Upper bound = bump the last character. Redis's own idiom is
        # ZRANGEBYLEX key "[prefix" "[prefix\xff", which has the same
        # off-by-one hazard on multi-byte terms — see notebook 3.
        upper = prefix[:-1] + chr(ord(prefix[-1]) + 1)
        lo = bisect.bisect_left(self._terms, prefix)
        hi = bisect.bisect_left(self._terms, upper)
        window = ((t, self._score[t]) for t in self._terms[lo:hi])
        return nlargest(k, window, key=lambda x: x[1])

z = ZSet()
for term, n in [
    ("python", 900), ("python tutorial", 500), ("pyramid", 100),
    ("google", 800), ("google docs", 400), ("golang", 200),
    ("new york", 700), ("new york times", 550), ("news", 600),
    ("netflix", 650),
]:
    z.zincrby(term, n)

print("prefix 'goo' →", z.zrangebylex_prefix("goo", k=3))
print("prefix 'new' →", z.zrangebylex_prefix("new", k=3))
print()
# Live update — no rebuild needed.
z.zincrby("google", 500)
print("after google surge →", z.zrangebylex_prefix("go", k=3))

## Trie vs Redis sorted-set — pick your poison

| | Trie + top-K | Redis ZSET + `ZRANGEBYLEX` |
|---|---|---|
| Query time | **O(L)** | O(log N + R) |
| Updates | Batch rebuild (5–15 min) | **Live** (`ZINCRBY`) |
| Memory | Higher (many nodes + cached top-K) | Lower |
| Ops burden | You run the aggregator + trie service | Just Redis |
| Scale ceiling | 100M+ terms, 100k+ QPS per shard | ~10M terms, few k QPS |
| When to pick | Big, hot, latency-critical | Small/medium or "just ship it" |

If you're starting from zero, **start with the sorted-set version**.
Swap to a trie service only when you hit its ceiling.

**Where the sorted-set version actually breaks**, so you recognise it early:

- **Short prefixes.** `ZRANGEBYLEX` finds the range in O(log N), but you still
  have to rank R members, and for a 1-character prefix R is a large slice of the
  whole set. Notebook 3 measured this: 4× slower than a trie on 3-char prefixes,
  **939× slower** on 1-char prefixes. Mitigation: keep a separate small ZSET of
  precomputed answers for the few thousand hottest short prefixes — at which
  point you have reinvented the top-K cache, just without the trie.
- **Live `ZINCRBY` is not actually free.** It writes on the read path's data
  structure, so a popularity surge and a query burst contend for the same keys.
- **No shared structure.** `"google"` and `"google docs"` are two independent
  members; the trie's prefix sharing is exactly what the ZSET gives up in
  exchange for being one Redis command.

## Further reading

- *Designing Data-Intensive Applications*, Kleppmann — indexing chapter (tries, LSM).
- Elasticsearch docs — [Completion Suggester](https://www.elastic.co/guide/en/elasticsearch/reference/current/search-suggesters.html#completion-suggester).
- Redis docs — [ZRANGEBYLEX](https://redis.io/commands/zrangebylex/).
- Google's Steve Yegge "search at Google" talks — flavor of how ranking grows.
- `references/designgurus.md` in this lab — source lessons feeding this material.
